# Somnotate Scoring and Inspection
Score EDF recordings by subject and date, then save predictions.

In [3]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from utils.scoring import score_recordings
from utils.paths import find_recordings
from utils.somnotate_pipeline.data_io import load_raw_signals
from utils.testing import plot_testing_comparison
from utils.config import MODEL_TO_OUTPUT_LABEL
import pandas as pd
%matplotlib qt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
model_name = "test_model_2"
repo_root = Path.cwd().parent
model_path = (
    repo_root
    / "data"
    / "hypnose_eeg"
    / "derivatives"
    / "somnotate_training"
    / model_name
    / "model.pickle"
)
subjids = [53]
dates = [20260429]
# date_range = (20260501, 20260510)

outputs = score_recordings(
    subjids=subjids,
    dates=dates,
    model_path=model_path,
    repo_root=repo_root,
)

for path in outputs:
    print("Saved predictions:", path)

[sub-053 ses-011 date-20260429] sub-053_ses-011_recording-001.edf
  Duration: 147.5 h | missing: 124.1 h (84.1%) | longest gap: 124.1 h
  Strategy: mask_inline — score the kept range as one chunk, mark middle gaps as undefined post-hoc
  Segments (4):
    [       0 ->    79239 s]  signal     (22.0 h)  (within chunk 1)
    [   79239 ->    79250 s]  gap        (11 s)
    [   79250 ->    84350 s]  signal     (1.4 h)  (within chunk 1)
    [   84350 ->   531090 s]  gap        (124.1 h)
Saved predictions: /Volumes/harris/hypnose/hypnose_eeg/derivatives/sub-053_id-366/ses-011_date-20260429/saved_results/sub-053_ses-011_recording-001_somnotate_predictions.parquet


In [3]:
# Visualize a single recording (one subject and one date).
viz_subjid = "53"
viz_date = "20260429"
recordings = find_recordings(repo_root, [viz_subjid], dates=[viz_date])
if not recordings:
    raise ValueError("No recordings found for the selected subject/date")
recording = recordings[0]

raw_signals = load_raw_signals(str(recording.edf_path), ["EEG EEG1A-B", "EEG EEG2A-B", "EMG EMG"])
pred_path = recording.output_dir / f"{recording.edf_path.stem}_somnotate_predictions.parquet"
pred_df = pd.read_parquet(pred_path)
if "label_model" in pred_df.columns:
    somnotate_vec = pred_df["label_model"].to_numpy(dtype=int)
else:
    inverse_map = {value: key for key, value in MODEL_TO_OUTPUT_LABEL.items()}
    somnotate_vec = pred_df["label"].map(lambda v: inverse_map.get(int(v), 0)).to_numpy(dtype=int)

fig, viewer = plot_testing_comparison(
    raw_signals,
    sampling_rate_hz=512,
    somnotate_vec=somnotate_vec,
    manual_vectors={},
)